# Top sequence selection


## Setup


In [11]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd


def repo_root_from_here() -> Path:
    here = Path.cwd().resolve()
    for path in (here, *here.parents):
        if (path / "window_sequence_analysis").is_dir():
            return path
    raise FileNotFoundError("Could not locate the repository root.")


def resolve_data_dir(root: Path, data_dir: str) -> Path:
    path = Path(data_dir)
    resolved = path if path.is_absolute() else (root / path)
    resolved = resolved.resolve()
    if not resolved.is_dir():
        raise FileNotFoundError(f"Data directory not found: {resolved}")
    return resolved


def latest_profile_csv(data_dir: Path) -> Path:
    filename = "window_sequence_profiles.csv"
    results_root = data_dir / "results"
    run_profiles = list((results_root / "runs").glob(f"*/{filename}"))
    if run_profiles:
        return max(run_profiles, key=lambda path: path.stat().st_mtime_ns)
    legacy_profile = results_root / filename
    if legacy_profile.is_file():
        return legacy_profile
    raise FileNotFoundError(
        f"No {filename} found in {results_root / 'runs'} or {results_root}."
    )


ROOT = repo_root_from_here()
DATA_DIR = "data/skin_samples"
DATA_PATH = resolve_data_dir(ROOT, DATA_DIR)
PROFILES_CSV = latest_profile_csv(DATA_PATH)
SEQUENCES_CSV = DATA_PATH / "sequences.csv"
TISSUE_TSV = DATA_PATH / "ms_tissue.tsv"

CLUSTER_COL = "single_cell_expression_cluster"
GENE_NAME_COL = "GN"
DELIMITER_COL = "profile_delimiter"
SIGMA_COLUMN = "hyperplane_distance_mean_profile"
UNASSIGNED_LABEL = "(unassigned)"
SIGMA_ACTIVITY_THRESHOLD = 0.0
DNGC_MIN_THRESHOLD = 0.01
TOP_N_CLUSTERS = 10
SEQUENCE_TOP_N = 20
WRITE_TOP_SEQUENCES_CSV = True
TOP_SEQUENCES_CSV = DATA_PATH / "results" / "top_sequences_by_dngc.csv"

print("ROOT:", ROOT)
print("profiles:", PROFILES_CSV)
print("sequences:", SEQUENCES_CSV)
print("tissue table:", TISSUE_TSV)


ROOT: /mnt/c/Users/bioin/Documents/Peptide-Anti-microbial-Properties-Prediction
profiles: /mnt/c/Users/bioin/Documents/Peptide-Anti-microbial-Properties-Prediction/data/skin_samples/results/runs/2026-09-16_20-47/window_sequence_profiles.csv
sequences: /mnt/c/Users/bioin/Documents/Peptide-Anti-microbial-Properties-Prediction/data/skin_samples/sequences.csv
tissue table: /mnt/c/Users/bioin/Documents/Peptide-Anti-microbial-Properties-Prediction/data/skin_samples/ms_tissue.tsv


## Load and sort by cluster

Profiles are grouped on `single_cell_expression_cluster`. Unassigned proteins are left out, and the table is ordered by cluster number.


In [12]:
def cluster_label(value: object) -> str:
    text = "" if pd.isna(value) else str(value).strip()
    return text or UNASSIGNED_LABEL


def cluster_sort_key(name: str) -> tuple[int, int, str]:
    match = re.match(r"Cluster\s+(\d+)", name)
    if match:
        return (0, int(match.group(1)), name)
    if name == UNASSIGNED_LABEL:
        return (2, 0, name)
    return (1, 0, name)


def load_cluster_table(path: Path) -> pd.DataFrame:
    if not path.is_file():
        raise FileNotFoundError(f"Window profile CSV not found: {path}")
    frame = pd.read_csv(path)
    required = {CLUSTER_COL, GENE_NAME_COL, DELIMITER_COL, SIGMA_COLUMN}
    missing = sorted(required - set(frame.columns))
    if missing:
        raise ValueError(f"Profile CSV missing columns: {missing}")
    if frame.empty:
        raise ValueError(f"Window profile CSV is empty: {path}")
    labels = frame[CLUSTER_COL].map(cluster_label)
    kept = frame.loc[labels != UNASSIGNED_LABEL].copy()
    kept[CLUSTER_COL] = labels.loc[kept.index].to_numpy()
    kept["_cluster_key"] = kept[CLUSTER_COL].map(cluster_sort_key)
    ordered = kept.sort_values("_cluster_key", kind="mergesort").drop(columns="_cluster_key")
    return ordered.reset_index(drop=True)


profiles = load_cluster_table(PROFILES_CSV)
print(f"Loaded {len(profiles):,} proteins in {profiles[CLUSTER_COL].nunique():,} clusters")


Loaded 4,601 proteins in 110 clusters


## Skin intensity

`Gene name` in `ms_tissue.tsv` is matched to `GN`. Only Tissue `skin` is used. A blank skin intensity is stored as 0, and a gene missing from the skin table is left as missing.


In [13]:
def _skin_intensity_value(intensity: object) -> float:
    value = pd.to_numeric(intensity, errors="coerce")
    if pd.isna(value):
        return 0.0
    return float(value)


def load_skin_intensity(path: Path) -> dict[str, float]:
    if not path.is_file():
        raise FileNotFoundError(f"Tissue table not found: {path}")
    frame = pd.read_csv(path, sep="\t", usecols=["Gene name", "Tissue", "Intensity"])
    skin = frame.loc[frame["Tissue"].astype(str).str.strip().str.casefold() == "skin"]
    scores: dict[str, float] = {}
    for gene_name, intensity in zip(skin["Gene name"], skin["Intensity"], strict=True):
        name = "" if pd.isna(gene_name) else str(gene_name).strip()
        if not name:
            continue
        value = _skin_intensity_value(intensity)
        previous = scores.get(name)
        if previous is None:
            scores[name] = value
            continue
        both_present = np.isfinite(previous) and np.isfinite(value)
        if both_present and previous != value:
            raise ValueError(f"{name} has conflicting skin intensities: {previous} and {value}.")
        if not np.isfinite(previous) and np.isfinite(value):
            scores[name] = value
    return scores


def append_skin_intensity(frame: pd.DataFrame, scores: dict[str, float]) -> pd.DataFrame:
    out = frame.copy()
    genes = out[GENE_NAME_COL].map(lambda value: "" if pd.isna(value) else str(value).strip())
    out["skin_intensity"] = genes.map(scores)
    return out


skin_intensity_by_gene = load_skin_intensity(TISSUE_TSV)
profiles = append_skin_intensity(profiles, skin_intensity_by_gene)
matched = int(profiles["skin_intensity"].notna().sum())
print(f"Skin genes in tissue table: {len(skin_intensity_by_gene):,}")
print(f"Proteins with a skin intensity: {matched:,}")
print(f"Proteins missing from the skin table: {len(profiles) - matched:,}")


Skin genes in tissue table: 19,295
Proteins with a skin intensity: 4,568
Proteins missing from the skin table: 33


## Weighted DNGC

Clusters are ranked by weighted DNGC, highest first. For each protein, DNGC is the mean excess σNGC on residues with σNGC > 0. A protein contributes `DNGC × skin intensity`, and the cluster score is that sum divided by the sum of the intensities. Proteins with DNGC below 0.01, and proteins without a skin intensity, are left out of the weight. Clusters with no usable intensity are placed last.


In [14]:
def parse_profile(text: object, delimiter: str = ";") -> np.ndarray:
    parts = str(text).split(delimiter or ";")
    values = np.empty(len(parts), dtype=np.float64)
    for index, part in enumerate(parts):
        token = part.strip()
        values[index] = np.nan if not token or token.lower() == "nan" else float(token)
    return values


def protein_dngc(text: object, delimiter: str) -> float:
    sigma = parse_profile(text, delimiter)
    active = np.isfinite(sigma) & (sigma > SIGMA_ACTIVITY_THRESHOLD)
    s_score = int(np.count_nonzero(active))
    if s_score == 0:
        return 0.0
    excess = sigma[active] - SIGMA_ACTIVITY_THRESHOLD
    return float(np.sum(excess) / s_score)


def _cluster_order(name: str) -> int:
    kind, number, _label = cluster_sort_key(name)
    return number if kind == 0 else 10**9


def clusters_by_weighted_dngc(frame: pd.DataFrame) -> pd.DataFrame:
    delimiters = frame[DELIMITER_COL].fillna(";").astype(str)
    dngc = np.fromiter(
        (
            protein_dngc(text, delimiter)
            for text, delimiter in zip(frame[SIGMA_COLUMN], delimiters, strict=True)
        ),
        dtype=np.float64,
        count=len(frame),
    )
    intensity = pd.to_numeric(frame["skin_intensity"], errors="coerce").to_numpy(dtype=np.float64)
    usable = np.isfinite(intensity) & (dngc >= DNGC_MIN_THRESHOLD)
    work = pd.DataFrame(
        {
            CLUSTER_COL: frame[CLUSTER_COL].to_numpy(),
            "usable": usable,
            "weight": np.where(usable, intensity, 0.0),
            "weighted_part": np.where(usable, dngc * intensity, 0.0),
        }
    )
    grouped = work.groupby(CLUSTER_COL, sort=False)
    counts = grouped.size()
    weighted_counts = grouped["usable"].sum()
    total_weight = grouped["weight"].sum()
    score = grouped["weighted_part"].sum() / total_weight.replace(0, np.nan)
    table = pd.DataFrame(
        {
            CLUSTER_COL: counts.index,
            "protein_count": counts.to_numpy(),
            "weighted_protein_count": weighted_counts.reindex(counts.index).to_numpy(),
            "weighted_dngc": score.reindex(counts.index).to_numpy(),
        }
    )
    table["_missing"] = ~np.isfinite(table["weighted_dngc"])
    table["_score"] = table["weighted_dngc"].fillna(0.0)
    table["_cluster_order"] = [_cluster_order(name) for name in table[CLUSTER_COL]]
    ordered = table.sort_values(
        ["_missing", "_score", "_cluster_order", CLUSTER_COL],
        ascending=[True, False, True, True],
        kind="mergesort",
    )
    return ordered.drop(columns=["_missing", "_score", "_cluster_order"]).reset_index(drop=True)


def format_weighted_dngc(value: float) -> str:
    if not np.isfinite(value):
        return "n/a"
    return f"{value:.4f}"


def print_top_clusters(table: pd.DataFrame, top_n_clusters: int) -> None:
    top = table.head(top_n_clusters)
    print(f"Top {len(top)} clusters by weighted DNGC")
    for rank, row in enumerate(top.itertuples(index=False), start=1):
        print(
            f"{rank:>2}. {getattr(row, CLUSTER_COL)}\n"
            f"    weighted DNGC = {format_weighted_dngc(float(row.weighted_dngc))}"
            f"    proteins = {int(row.protein_count)}"
            f"    weighted n = {int(row.weighted_protein_count)}"
        )


ranked_clusters = clusters_by_weighted_dngc(profiles)
print_top_clusters(ranked_clusters, TOP_N_CLUSTERS)


Top 10 clusters by weighted DNGC
 1. Cluster 30: Epididymal principal cells - Sperm storage microenvironment
    weighted DNGC = 1.5798    proteins = 4    weighted n = 1
 2. Cluster 106: Spermatogonia - Spermatogonial differentiation
    weighted DNGC = 1.0057    proteins = 7    weighted n = 1
 3. Cluster 44: Plasma cells - Antibody production & secretion
    weighted DNGC = 0.6917    proteins = 37    weighted n = 10
 4. Cluster 22: Non-specific - Basic cellular functions
    weighted DNGC = 0.6289    proteins = 37    weighted n = 15
 5. Cluster 43: Sertoli cells - Spermatogenesis microenvironment
    weighted DNGC = 0.5817    proteins = 7    weighted n = 2
 6. Cluster 65: Non-specific - Cell growth & division
    weighted DNGC = 0.5497    proteins = 32    weighted n = 19
 7. Cluster 8: Non-specific - Secretory pathway
    weighted DNGC = 0.5113    proteins = 71    weighted n = 36
 8. Cluster 27: Non-specific - Cell proliferation
    weighted DNGC = 0.5104    proteins = 44    weighted 

## Top sequences by DNGC

Proteins are ranked by DNGC, highest first. DNGC is the mean excess σNGC on residues with σNGC > 0. The DNGC minimum used for cluster weights is not applied here.


In [15]:
def protein_dngc_scores(frame: pd.DataFrame) -> np.ndarray:
    delimiters = frame[DELIMITER_COL].fillna(";").astype(str)
    return np.fromiter(
        (
            protein_dngc(text, delimiter)
            for text, delimiter in zip(frame[SIGMA_COLUMN], delimiters, strict=True)
        ),
        dtype=np.float64,
        count=len(frame),
    )


def sequences_by_dngc(frame: pd.DataFrame, sequence_by_id: pd.Series) -> pd.DataFrame:
    required = {"id", "name", "skin_intensity"}
    missing = sorted(required - set(frame.columns))
    if missing:
        raise ValueError(f"Profile table missing columns: {missing}")
    ids = frame["id"].map(lambda value: "" if pd.isna(value) else str(value).strip())
    table = pd.DataFrame(
        {
            "id": ids.to_numpy(),
            GENE_NAME_COL: frame[GENE_NAME_COL].to_numpy(),
            "name": frame["name"].to_numpy(),
            "sequence": ids.map(sequence_by_id).to_numpy(),
            "skin_intensity": pd.to_numeric(frame["skin_intensity"], errors="coerce").to_numpy(),
            CLUSTER_COL: frame[CLUSTER_COL].to_numpy(),
            "dngc": protein_dngc_scores(frame),
        }
    )
    ordered = table.sort_values(["dngc", "id"], ascending=[False, True], kind="mergesort")
    return ordered.reset_index(drop=True)


def load_sequence_by_id(path: Path) -> pd.Series:
    if not path.is_file():
        raise FileNotFoundError(f"Sequence table not found: {path}")
    frame = pd.read_csv(path, usecols=["id", "sequence"])
    ids = frame["id"].map(lambda value: "" if pd.isna(value) else str(value).strip())
    sequences = frame["sequence"].map(lambda value: "" if pd.isna(value) else str(value).strip())
    table = pd.DataFrame({"id": ids, "sequence": sequences})
    table = table.loc[table["id"] != ""]
    counts = table.groupby("id", sort=False)["sequence"].nunique()
    conflicts = counts[counts > 1]
    if not conflicts.empty:
        name = str(conflicts.index[0])
        raise ValueError(f"{name} has conflicting sequences.")
    return table.drop_duplicates("id", keep="first").set_index("id")["sequence"]


def _ranked_top(table: pd.DataFrame, top_n: int) -> pd.DataFrame:
    top = table.head(top_n).copy()
    top.insert(0, "rank", np.arange(1, len(top) + 1))
    return top


from IPython.display import display


def top_sequence_table(table: pd.DataFrame, top_n: int) -> pd.DataFrame:
    top = _ranked_top(table, top_n)
    top["dngc"] = top["dngc"].map(lambda value: f"{float(value):.4f}")
    shown = top.drop(columns=["id", GENE_NAME_COL, "sequence", "skin_intensity"])
    return shown.rename(columns={CLUSTER_COL: "cluster", "dngc": "DNGC"})


def top_sequence_export(table: pd.DataFrame, top_n: int) -> pd.DataFrame:
    top = _ranked_top(table, top_n)
    top["dngc"] = top["dngc"].map(lambda value: f"{float(value):.4f}")
    exported = top.rename(columns={CLUSTER_COL: "cluster", "dngc": "DNGC"})
    return exported[
        ["rank", "id", GENE_NAME_COL, "name", "sequence", "skin_intensity", "cluster", "DNGC"]
    ]


def write_top_sequence_csv(table: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(path, index=False)


sequence_by_id = load_sequence_by_id(SEQUENCES_CSV)
ranked_sequences = sequences_by_dngc(profiles, sequence_by_id)
top_sequences = top_sequence_table(ranked_sequences, SEQUENCE_TOP_N)
print(f"Top {len(top_sequences)} sequences by DNGC")
display(top_sequences.style.hide(axis="index"))
if WRITE_TOP_SEQUENCES_CSV:
    write_top_sequence_csv(top_sequence_export(ranked_sequences, SEQUENCE_TOP_N), TOP_SEQUENCES_CSV)
    print(f"Wrote {TOP_SEQUENCES_CSV}")


Top 20 sequences by DNGC


rank,name,cluster,DNGC
1,Cathelicidin antimicrobial peptide,Cluster 30: Epididymal principal cells - Sperm storage microenvironment,1.5798
2,N(6)-adenosine-methyltransferase non-catalytic subunit METTL14,Cluster 7: Non-specific - Transcription,1.4999
3,Lethal(2) giant larvae protein homolog 1,Cluster 80: Oligodendrocytes - Myelination,1.4860
4,Histone H4,Cluster 66: Erythroid cells - Erythroid differentiation and maturation,1.4339
5,Sorting nexin-24,Cluster 72: Macrophages - Phagocytosis & lysosomal degradation,1.3759
6,H/ACA ribonucleoprotein complex subunit 1,Cluster 17: Non-specific - RNA processing,1.3467
7,H/ACA ribonucleoprotein complex subunit 2,Cluster 85: Squamous epithelial cells - Keratinization,1.3280
8,Large ribosomal subunit protein eL32,Cluster 15: Non-specific - Protein processing,1.2594
9,RNA-binding protein 8A,Cluster 15: Non-specific - Protein processing,1.2531
10,Large ribosomal subunit protein eL13,Cluster 15: Non-specific - Protein processing,1.2390


Wrote /mnt/c/Users/bioin/Documents/Peptide-Anti-microbial-Properties-Prediction/data/skin_samples/results/top_sequences_by_dngc.csv
